# Task 8 — AI Customer Support Ticket Automation System
Workflow: classify the request (category + priority) -> generate a professional response -> automatically run the matching backend action -> save the ticket + log the activity.

## Imports

In [1]:
!pip install -q transformers accelerate bitsandbytes

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.8 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
import json
import os
import datetime
import itertools

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)

## Model Loading

In [3]:
model_name = "Qwen/Qwen2-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Step 1 — Classify The Request (LLM)
Asks the model for ONLY `CATEGORY=` and `PRIORITY=` lines (same reliable key=value parsing style as `Automation_Project_Managment.ipynb`), then validates the result against the allowed values — this is the 'handle invalid classification' requirement.

In [4]:
ALLOWED_CATEGORIES = ["Technical", "Billing", "Account", "General"]
ALLOWED_PRIORITIES = ["Low", "Medium", "High"]

def classify_ticket(message: str):
    prompt = f"""
You are a customer support triage assistant.

Classify the customer's message below.

Return ONLY in this exact format, nothing else:
CATEGORY=<Technical|Billing|Account|General>
PRIORITY=<Low|Medium|High>

Do NOT explain. Do NOT add notes.

Customer message:
{message}

Answer:
"""

    result = pipe(
        prompt,
        max_new_tokens=30,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )[0]["generated_text"]

    generated = result[len(prompt):].strip()

    category, priority = None, None
    for line in generated.splitlines():
        line = line.strip()
        if line.startswith("CATEGORY="):
            category = line.replace("CATEGORY=", "").strip()
        elif line.startswith("PRIORITY="):
            priority = line.replace("PRIORITY=", "").strip()

    # Handle invalid / missing classification
    if category not in ALLOWED_CATEGORIES:
        print(f"Warning: invalid or missing category '{category}' -> defaulting to 'General'.")
        category = "General"

    if priority not in ALLOWED_PRIORITIES:
        print(f"Warning: invalid or missing priority '{priority}' -> defaulting to 'Medium'.")
        priority = "Medium"

    return category, priority

## Step 2 — Generate A Professional Response (LLM)
Separate call (same pattern as `generate_report` in `Automation_Report.ipynb`) so the free-text response isn't mangled by the line-filtering used for classification.

In [5]:
def generate_response(message: str, category: str) -> str:
    prompt = f"""You are a professional customer support agent.
Write a short, polite, professional response (2-3 sentences) to the customer below.
The issue category is: {category}.
Return only the response text, no extra explanation.

Customer message:
{message}

Response:
"""

    result = pipe(prompt, max_new_tokens=120, do_sample=False, eos_token_id=tokenizer.eos_token_id)[0]["generated_text"]

    if "Response:" in result:
        response = result.split("Response:")[-1].strip()
    else:
        response = result[len(prompt):].strip()

    return response

## Step 3 — API-Like Ticket Operations
One backend function per category, simulating the system each kind of ticket would really be routed to.

In [6]:
def technical_support_api(ticket: dict) -> dict:
    print("Technical Support API called -> ticket routed to the engineering queue.")
    return {"action": "routed_to_engineering", "queue": "Technical Support"}

def billing_api(ticket: dict) -> dict:
    print("Billing API called -> ticket routed to the billing department.")
    return {"action": "routed_to_billing", "queue": "Billing"}

def account_api(ticket: dict) -> dict:
    print("Account API called -> account verification workflow triggered.")
    return {"action": "account_verification_triggered", "queue": "Account Services"}

def general_api(ticket: dict) -> dict:
    print("General Inquiry API called -> logged for the general support team.")
    return {"action": "logged_general_inquiry", "queue": "General Support"}

CATEGORY_ACTIONS = {
    "Technical": technical_support_api,
    "Billing": billing_api,
    "Account": account_api,
    "General": general_api,
}

def execute_category_action(ticket: dict) -> dict:
    action_fn = CATEGORY_ACTIONS.get(ticket["category"], general_api)
    return action_fn(ticket)

## Step 4 — Save Ticket + Log Activity
Saves each ticket as its own JSON file and appends one line to a shared activity log, wrapped in try/except so a disk error doesn't crash the whole workflow.

In [7]:
TICKETS_DIR = "/content/tickets"

def save_ticket_api(ticket: dict) -> dict:
    try:
        os.makedirs(TICKETS_DIR, exist_ok=True)
        filepath = os.path.join(TICKETS_DIR, f"ticket_{ticket['ticket_id']}.json")

        with open(filepath, "w") as f:
            json.dump(ticket, f, indent=2)

        log_line = (
            f"[{ticket['timestamp']}] Ticket {ticket['ticket_id']} | "
            f"{ticket['category']} | {ticket['priority']} | saved\n"
        )
        with open(os.path.join(TICKETS_DIR, "activity_log.txt"), "a") as f:
            f.write(log_line)

        return {"status": "saved", "file": filepath}

    except Exception as e:
        return {"status": "error", "message": str(e)}

## Full Workflow
Ties every step together and handles the two required error cases: empty input (checked before calling the LLM at all) and invalid classification (handled inside `classify_ticket`).

In [8]:
ticket_counter = itertools.count(1)

def process_ticket(message: str) -> dict:
    print("\n" + "=" * 60)
    print("New Support Request")
    print("=" * 60)

    # Handle empty input
    if not message or not message.strip():
        print("Error: empty customer message.")
        return {"status": "error", "message": "Empty customer message."}

    ticket_id = next(ticket_counter)
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"Message: {message}")

    # Step 1: classify
    category, priority = classify_ticket(message)
    print(f"Category: {category} | Priority: {priority}")

    # Step 2: generate response
    response_text = generate_response(message, category)

    ticket = {
        "ticket_id": ticket_id,
        "timestamp": timestamp,
        "message": message,
        "category": category,
        "priority": priority,
        "response": response_text,
        "status": "open"
    }

    # Step 3: automated action based on category
    ticket["action_result"] = execute_category_action(ticket)

    # Step 4: save + log
    save_result = save_ticket_api(ticket)
    ticket["save_result"] = save_result
    ticket["status"] = "processed" if save_result["status"] == "saved" else "processing_error"

    # Step 5: display final ticket
    print("\nFinal Ticket")
    print("-" * 60)
    for k, v in ticket.items():
        print(f"{k}: {v}")

    return ticket

## Tests

In [9]:
process_ticket("My internet keeps disconnecting every few minutes, please help urgently.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'eos_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



New Support Request
Message: My internet keeps disconnecting every few minutes, please help urgently.


Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Category: General | Priority: Medium
General Inquiry API called -> logged for the general support team.

Final Ticket
------------------------------------------------------------
ticket_id: 1
timestamp: 2026-10-01 16:13:56
message: My internet keeps disconnecting every few minutes, please help urgently.
category: General
priority: Medium
response: I'm sorry to hear that you're experiencing this issue. Can you please provide more details about your internet connection and any troubleshooting steps you've already tried? I'll do my best to assist you in resolving this problem. Thank you for choosing our service.
status: processed
action_result: {'action': 'logged_general_inquiry', 'queue': 'General Support'}
save_result: {'status': 'saved', 'file': '/content/tickets/ticket_1.json'}


{'ticket_id': 1,
 'timestamp': '2026-10-01 16:13:56',
 'message': 'My internet keeps disconnecting every few minutes, please help urgently.',
 'category': 'General',
 'priority': 'Medium',
 'response': "I'm sorry to hear that you're experiencing this issue. Can you please provide more details about your internet connection and any troubleshooting steps you've already tried? I'll do my best to assist you in resolving this problem. Thank you for choosing our service.",
 'status': 'processed',
 'action_result': {'action': 'logged_general_inquiry',
  'queue': 'General Support'},
 'save_result': {'status': 'saved', 'file': '/content/tickets/ticket_1.json'}}

In [10]:
process_ticket("I was charged twice for my last subscription payment.")

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



New Support Request
Message: I was charged twice for my last subscription payment.


Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Category: General | Priority: High
General Inquiry API called -> logged for the general support team.

Final Ticket
------------------------------------------------------------
ticket_id: 2
timestamp: 2026-10-01 16:14:04
message: I was charged twice for my last subscription payment.
category: General
priority: High
response: Hello, I'm sorry to hear that you were charged twice for your last subscription payment. Our team will review this matter and provide an appropriate resolution as soon as possible. Please keep in mind that we value your feedback and apologize for any inconvenience caused by this error. Thank you for choosing our service. Have a great day!
status: processed
action_result: {'action': 'logged_general_inquiry', 'queue': 'General Support'}
save_result: {'status': 'saved', 'file': '/content/tickets/ticket_2.json'}


{'ticket_id': 2,
 'timestamp': '2026-10-01 16:14:04',
 'message': 'I was charged twice for my last subscription payment.',
 'category': 'General',
 'priority': 'High',
 'response': "Hello, I'm sorry to hear that you were charged twice for your last subscription payment. Our team will review this matter and provide an appropriate resolution as soon as possible. Please keep in mind that we value your feedback and apologize for any inconvenience caused by this error. Thank you for choosing our service. Have a great day!",
 'status': 'processed',
 'action_result': {'action': 'logged_general_inquiry',
  'queue': 'General Support'},
 'save_result': {'status': 'saved', 'file': '/content/tickets/ticket_2.json'}}

In [11]:
process_ticket("I can't log into my account, it says my password is wrong even after reset.")

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



New Support Request
Message: I can't log into my account, it says my password is wrong even after reset.


Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Category: General | Priority: Medium
General Inquiry API called -> logged for the general support team.

Final Ticket
------------------------------------------------------------
ticket_id: 3
timestamp: 2026-10-01 16:14:12
message: I can't log into my account, it says my password is wrong even after reset.
category: General
priority: Medium
response: Hello, I'm sorry to hear that you're having trouble logging in. Please try resetting your password again and then try logging in once more. If this still doesn't work, please contact our technical support team for further assistance. Thank you for choosing our service. Have a great day!
status: processed
action_result: {'action': 'logged_general_inquiry', 'queue': 'General Support'}
save_result: {'status': 'saved', 'file': '/content/tickets/ticket_3.json'}


{'ticket_id': 3,
 'timestamp': '2026-10-01 16:14:12',
 'message': "I can't log into my account, it says my password is wrong even after reset.",
 'category': 'General',
 'priority': 'Medium',
 'response': "Hello, I'm sorry to hear that you're having trouble logging in. Please try resetting your password again and then try logging in once more. If this still doesn't work, please contact our technical support team for further assistance. Thank you for choosing our service. Have a great day!",
 'status': 'processed',
 'action_result': {'action': 'logged_general_inquiry',
  'queue': 'General Support'},
 'save_result': {'status': 'saved', 'file': '/content/tickets/ticket_3.json'}}

In [12]:
process_ticket("What are your support hours?")

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



New Support Request
Message: What are your support hours?


Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Category: General | Priority: Medium
General Inquiry API called -> logged for the general support team.

Final Ticket
------------------------------------------------------------
ticket_id: 4
timestamp: 2026-10-01 16:14:19
message: What are your support hours?
category: General
priority: Medium
response: We're sorry, but we don't have that information available. Please refer to our website for more details on our support hours. Thank you for contacting us!
status: processed
action_result: {'action': 'logged_general_inquiry', 'queue': 'General Support'}
save_result: {'status': 'saved', 'file': '/content/tickets/ticket_4.json'}


{'ticket_id': 4,
 'timestamp': '2026-10-01 16:14:19',
 'message': 'What are your support hours?',
 'category': 'General',
 'priority': 'Medium',
 'response': "We're sorry, but we don't have that information available. Please refer to our website for more details on our support hours. Thank you for contacting us!",
 'status': 'processed',
 'action_result': {'action': 'logged_general_inquiry',
  'queue': 'General Support'},
 'save_result': {'status': 'saved', 'file': '/content/tickets/ticket_4.json'}}

In [13]:
# Error case: empty input
process_ticket("")


New Support Request
Error: empty customer message.


{'status': 'error', 'message': 'Empty customer message.'}

## Run A Continuous Support Loop
Interactive loop for manual testing. Type `exit` to stop.

In [ ]:
while True:
    msg = input("\nCustomer message (or 'exit'): ").strip()

    if msg.lower() == "exit":
        print("Session ended.")
        break

    process_ticket(msg)


Customer message (or 'exit'):  i have a problem


Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



New Support Request
Message: i have a problem


Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Category: General | Priority: High
General Inquiry API called -> logged for the general support team.

Final Ticket
------------------------------------------------------------
ticket_id: 5
timestamp: 2026-10-01 16:23:38
message: i have a problem
category: General
priority: High
response: I'm sorry to hear about your issue. Please provide more details so I can assist you better. Thank you for choosing our service. Have a great day!
status: processed
action_result: {'action': 'logged_general_inquiry', 'queue': 'General Support'}
save_result: {'status': 'saved', 'file': '/content/tickets/ticket_5.json'}
